# Phase 1 scratch: load and explore
CSV to DuckDB to parquet, then the basic facts about the data.

In [1]:
import sys, time
sys.path.insert(0, '../../src')
import load
load.build_base()
con = load.connect()
out = load.summary(con)
out['overall']

,transactions,fraud_txns,fraud_rate_pct,pct_with_identity,first_day,last_day,span_days,total_amt,fraud_amt
0,590540,20663.0,3.5,24.4,1,182,182,79738949.0,3083845.0


In [2]:
out['by_product']

,ProductCD,transactions,share_pct,fraud_rate_pct,pct_with_identity
0,W,439670,74.5,2.04,0.0
1,C,68519,11.6,11.69,90.8
2,R,37699,6.4,3.78,99.6
3,H,33024,5.6,4.77,99.6
4,S,11628,2.0,5.90,99.6


In [3]:
t = time.time()
df = con.sql('SELECT * FROM base').df()
print(f'full load: {time.time()-t:.1f}s, shape {df.shape}')

full load: 10.4s, shape (590540, 436)


Fraud rate by 30-day block, to sanity check the planned time split (train on days 1 to ~120, test after).

In [4]:
con.sql('''SELECT (day-1)//30 + 1 AS block, MIN(day) AS first_day, MAX(day) AS last_day,
       COUNT(*) AS n, ROUND(100*AVG(isFraud), 2) AS fraud_pct
FROM base GROUP BY 1 ORDER BY 1''').df()

,block,first_day,last_day,n,fraud_pct
0,1,1,30,134339,2.53
1,2,31,60,89399,4.00
2,3,61,90,92189,4.04
3,4,91,120,98615,3.95
4,5,121,150,83571,3.41
5,6,151,180,86934,3.42
6,7,181,182,5493,4.39


How often the identity-rebuild keys are missing (matters for Phase 2).

In [5]:
con.sql('''SELECT ROUND(100*AVG((addr1 IS NULL)::INT), 2) AS addr1_null_pct,
       ROUND(100*AVG((D1 IS NULL)::INT), 2) AS d1_null_pct,
       ROUND(100*AVG((P_emaildomain IS NULL)::INT), 2) AS p_email_null_pct,
       ROUND(100*AVG((DeviceInfo IS NULL)::INT), 2) AS device_info_null_pct
FROM base''').df()

,addr1_null_pct,d1_null_pct,p_email_null_pct,device_info_null_pct
0,11.13,0.21,15.99,79.91
